In [ ]:
# Example 1: Inspect the package's top-level public names + module file location.
import jupyter_server_mcp
import inspect

public_names = [n for n in dir(jupyter_server_mcp) if not n.startswith('_')]
print(f"Number of public names: {len(public_names)}")
for name in public_names[:20]:
    obj = getattr(jupyter_server_mcp, name)
    print(f"  {name}: {type(obj).__name__}")

pkg_file = inspect.getfile(jupyter_server_mcp)
print(f"\nPackage __init__ file: {pkg_file}")
assert pkg_file.endswith('__init__.py'), 'expected package __init__.py'


In [ ]:
# Example 2: Confirm core dependencies (fastmcp, jupyter_server) are importable alongside.
import fastmcp
import jupyter_server
import jupyter_server_mcp

print(f"fastmcp.__version__ = {getattr(fastmcp, '__version__', 'unknown')}")
print(f"jupyter_server.__version__ = {getattr(jupyter_server, '__version__', 'unknown')}")
print(f"jupyter_server_mcp loaded from = {jupyter_server_mcp.__path__[0]}")

assert fastmcp is not None
assert jupyter_server is not None


In [ ]:
# Example 3: Discover any ExtensionApp subclass the package exposes for jupyter_server.
import importlib
import pkgutil
import inspect

import jupyter_server_mcp
from jupyter_server.extension.application import ExtensionApp

found = []
for module_info in pkgutil.walk_packages(jupyter_server_mcp.__path__, prefix='jupyter_server_mcp.'):
    try:
        mod = importlib.import_module(module_info.name)
    except Exception:
        continue
    for name, obj in inspect.getmembers(mod, inspect.isclass):
        if obj is ExtensionApp:
            continue
        if issubclass(obj, ExtensionApp) and obj.__module__.startswith('jupyter_server_mcp'):
            found.append(f"{obj.__module__}.{obj.__name__}")

print(f"Discovered ExtensionApp subclasses: {found}")


In [ ]:
# Example 4: Check that the jupyter_server extension entry-point is registered.
import importlib.metadata as md

dist = md.distribution('jupyter-server-mcp')
print(f"Distribution: {dist.metadata['Name']} {dist.version}")

eps = list(dist.entry_points)
print(f"Total entry points: {len(eps)}")
for ep in eps:
    print(f"  group={ep.group} name={ep.name} value={ep.value}")

assert dist.version, 'distribution version should be non-empty'


In [ ]:
# Example 5: Look for an MCP server / FastMCP instance exposed by the package.
import importlib
import pkgutil
import inspect

import jupyter_server_mcp
from fastmcp import FastMCP

mcp_objects = []
modules_scanned = 0
for module_info in pkgutil.walk_packages(jupyter_server_mcp.__path__, prefix='jupyter_server_mcp.'):
    try:
        mod = importlib.import_module(module_info.name)
    except Exception:
        continue
    modules_scanned += 1
    for name, obj in inspect.getmembers(mod):
        if isinstance(obj, FastMCP):
            mcp_objects.append(f"{module_info.name}.{name}")

print(f"Modules scanned: {modules_scanned}")
print(f"FastMCP instances found: {mcp_objects}")
assert modules_scanned >= 1, 'expected at least one submodule scan'
